# 02 — TransformerLens Basics

Deeper tour of the API you'll use constantly: `utils.get_act_name`,
`circuitsvis` for attention, and the embedding/unembedding matrices.


## New shorthand in this notebook

(`d_model`, `d_vocab`, `resid`, `pos` etc. were covered in notebook 00's
glossary — this just adds what's new here.)

- **`W_` prefix** — TransformerLens names every learned weight matrix
  `W_<something>`. You'll meet `W_E` and `W_U` below; later notebooks add
  `W_Q`/`W_K`/`W_V`/`W_O` (the per-head attention projections) and
  `W_in`/`W_out` (the MLP's two linear layers). The prefix always just means
  "this is a weight matrix," never anything more.
- **`W_E`** — the **e**mbedding matrix. Row `i` is the starting
  residual-stream vector for token id `i`. Shape `[d_vocab, d_model]`
  because it has one row per vocabulary entry.
- **`W_U`** — the **u**nembedding matrix. Multiplying a residual-stream
  vector by `W_U` converts it into logits (one score per vocabulary token).
  Shape `[d_model, d_vocab]` — the exact transpose shape of `W_E`, because
  it's doing the reverse mapping.
- **`cv`** — the `circuitsvis` package (imported as `cv` by convention,
  same idea as `np` for numpy). Renders attention patterns as an interactive
  widget instead of a static heatmap.


In [1]:
import torch
from transformer_lens import HookedTransformer, utils
import circuitsvis as cv

torch.set_grad_enabled(False)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = HookedTransformer.from_pretrained("gpt2", device=device)


/home/keqingsimp/.conda/envs/mech-interp/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/tmp/ipykernel_67935/2148768489.py:2: DeprecationWarning: The 'utils' module has been deprecated. Please use 'transformer_lens.utilities' instead. Importing from utils.py will be removed in TransformerLens 4.0.
  from transformer_lens import HookedTransformer, utils


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 13712.63it/s]

Loaded pretrained model gpt2 into HookedTransformer


## `utils.get_act_name` — don't hand-write hook name strings

`get_act_name("pattern", layer)` builds `blocks.{layer}.attn.hook_pattern`
for you. Use this everywhere; it avoids typos and reads clearly.


In [2]:
text = "When Mary and John went to the store, John gave a drink to"
tokens = model.to_tokens(text)
str_tokens = model.to_str_tokens(text)
logits, cache = model.run_with_cache(tokens)

layer = 9
pattern = cache[utils.get_act_name("pattern", layer)]  # [batch, head, dest, src]
print(pattern.shape)


torch.Size([1, 12, 15, 15])


In [3]:
# circuitsvis gives an interactive, per-head attention viewer in the notebook
cv.attention.attention_patterns(tokens=str_tokens, attention=pattern[0])


## Embedding and unembedding

`model.W_E` maps token ids -> residual stream vectors (`[d_vocab, d_model]`).
`model.W_U` maps residual stream vectors -> logits (`[d_model, d_vocab]`).
Because the residual stream is additive, you can take *any* intermediate
residual stream vector and multiply it by `W_U` to see "what would the model
predict if it stopped here?" — that's the logit lens, next notebook.


In [4]:
print("W_E:", model.W_E.shape)
print("W_U:", model.W_U.shape)

# sanity check: greedy next-token prediction from the final logits
next_token_logits = logits[0, -1]
top5 = next_token_logits.topk(5).indices
print("top-5 next token predictions:", [model.to_string(t) for t in top5])


W_E: torch.Size([50257, 768])
W_U: torch.Size([768, 50257])
top-5 next token predictions: [' Mary', ' them', ' the', ' John', ' her']


## Next

`03_induction_heads.ipynb` — find the heads responsible for GPT-2's simplest
learned circuit: "if I've seen `[A][B]` before, and I see `[A]` again, predict `[B]`."
